In [5]:
!pip install gradio anthropic openai -q

In [6]:
from google.colab import files
uploaded = files.upload()  # upload deep_clf_nn.pt and deep_reg_nn.pt

Saving deep_clf_nn.pt to deep_clf_nn (1).pt
Saving deep_reg_nn.pt to deep_reg_nn (1).pt


In [7]:
api_key="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"

In [8]:
class LLMExplainer:
    def __init__(self, api_key, model="anthropic/claude-3-haiku"):
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        self.model = model

    def explain(self, diabetes_risk, bmi, inputs):
        prompt = f"""A patient has the following health indicators:
{inputs}

Model predictions:
- Diabetes Risk: {"High" if diabetes_risk else "Low"}
- Predicted BMI: {bmi:.1f}

In 3-4 sentences, explain what this means for the patient in plain English and suggest one actionable lifestyle tip."""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content


# Usage
explainer = LLMExplainer(api_key=api_key)

In [9]:
import torch
import torch.nn as nn
import gradio as gr
from openai import OpenAI

# ── Models ──────────────────────────────────────────────────
class DeepClassifierNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x).squeeze()

class DeepRegressorNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x).squeeze()

clf_model = DeepClassifierNN(21)
clf_model.load_state_dict(torch.load("deep_clf_nn.pt", map_location="cpu"))
clf_model.eval()

reg_model = DeepRegressorNN(20)
reg_model.load_state_dict(torch.load("deep_reg_nn.pt", map_location="cpu"))
reg_model.eval()

# ── LLM ─────────────────────────────────────────────────────
class LLMExplainer:
    def __init__(self, api_key, model="anthropic/claude-3-haiku"):
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        self.model = model

    def explain(self, diabetes_risk, bmi, inputs):
        prompt = f"""A patient has the following health indicators:
{inputs}

Model predictions:
- Diabetes Risk: {"High" if diabetes_risk else "Low"}
- Predicted BMI: {bmi:.1f}

In 3-4 sentences, explain what this means for the patient in plain English and suggest one actionable lifestyle tip."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

explainer = LLMExplainer(api_key=api_key)

# ── Predict ──────────────────────────────────────────────────
def predict(HighBP, HighChol, CholCheck, height_cm, weight_kg, Smoker, Stroke,
            HeartDiseaseorAttack, PhysActivity, Fruits, Veggies,
            HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth,
            MentHlth, PhysHlth, DiffWalk, Sex, Age, Income, Education):

    height_m = height_cm / 100
    BMI = round(weight_kg / (height_m ** 2), 1)

    print(f"BMI: {BMI}, raw logit: {clf_model(clf_input).item():.4f}")

    clf_input = torch.FloatTensor([[HighBP, HighChol, CholCheck, BMI, Smoker,
                                    Stroke, HeartDiseaseorAttack, PhysActivity,
                                    Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare,
                                    NoDocbcCost, GenHlth, MentHlth, PhysHlth,
                                    DiffWalk, Sex, Age, Income, Education]])

    reg_input = torch.FloatTensor([[HighBP, HighChol, CholCheck, Smoker, Stroke,
                                    HeartDiseaseorAttack, PhysActivity, Fruits,
                                    Veggies, HvyAlcoholConsump, AnyHealthcare,
                                    NoDocbcCost, GenHlth, MentHlth, PhysHlth,
                                    DiffWalk, Sex, Age, Income, Education]])

    with torch.no_grad():
        diabetes_risk = (clf_model(clf_input) > 0.5).item()
        predicted_bmi = reg_model(reg_input).item()

    inputs_str = f"Age: {Age}, Sex: {Sex}, BMI: {BMI}, Smoker: {Smoker}, PhysActivity: {PhysActivity}, GenHlth: {GenHlth}, Income: {Income}"
    explanation = explainer.explain(diabetes_risk, predicted_bmi, inputs_str)

    return (
        "⚠️ High Risk" if diabetes_risk else "✅ Low Risk",
        f"Your BMI: {BMI} | Predicted by model: {predicted_bmi:.1f}",
        explanation
    )

# ── UI ───────────────────────────────────────────────────────
with gr.Blocks(title="Do I Have Diabetes?") as app:
    gr.Markdown("# 🩺 Do I Have Diabetes?")
    gr.Markdown("Answer the questions below and click **Predict**.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📋 Basic Info")
            Age       = gr.Dropdown(
                choices=[(f"{18+i*5}–{22+i*5}", i+1) for i in range(13)],
                label="Age Group", value=5)
            Sex       = gr.Radio(choices=[("Female", 0), ("Male", 1)], label="Sex", value=0)
            height_cm = gr.Number(label="Height (cm)", value=170)
            weight_kg = gr.Number(label="Weight (kg)", value=70)
            Education = gr.Dropdown(
                choices=[("Never attended school", 1), ("Elementary", 2),
                         ("Some high school", 3), ("High school graduate", 4),
                         ("Some college", 5), ("College graduate", 6)],
                label="Education Level", value=4)
            Income    = gr.Dropdown(
                choices=[("< $10k", 1), ("$10–15k", 2), ("$15–20k", 3),
                         ("$20–25k", 4), ("$25–35k", 5), ("$35–50k", 6),
                         ("$50–75k", 7), ("> $75k", 8)],
                label="Annual Income", value=5)

        with gr.Column():
            gr.Markdown("### 🏥 Medical History")
            HighBP               = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="High Blood Pressure?", value=0)
            HighChol             = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="High Cholesterol?", value=0)
            CholCheck            = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Cholesterol checked in last 5 years?", value=1)
            Stroke               = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Ever had a Stroke?", value=0)
            HeartDiseaseorAttack = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Heart Disease or Attack?", value=0)
            DiffWalk             = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Difficulty Walking?", value=0)

        with gr.Column():
            gr.Markdown("### 🌿 Lifestyle")
            Smoker            = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Smoker?", value=0)
            HvyAlcoholConsump = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Heavy Alcohol Consumption?", value=0)
            PhysActivity      = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Physically Active?", value=1)
            Fruits            = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Eats Fruits Daily?", value=1)
            Veggies           = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Eats Vegetables Daily?", value=1)
            AnyHealthcare     = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Has Healthcare Coverage?", value=1)
            NoDocbcCost       = gr.Radio(choices=[("No", 0), ("Yes", 1)], label="Avoided Doctor due to Cost?", value=0)

            gr.Markdown("### 💬 How do you feel?")
            GenHlth  = gr.Dropdown(
                choices=[("Excellent", 1), ("Very Good", 2), ("Good", 3), ("Fair", 4), ("Poor", 5)],
                label="General Health", value=3)
            MentHlth = gr.Slider(0, 30, step=1, label="Days of Poor Mental Health (last 30 days)", value=0)
            PhysHlth = gr.Slider(0, 30, step=1, label="Days of Poor Physical Health (last 30 days)", value=0)

    btn = gr.Button("🔍 Predict", variant="primary", size="lg")

    with gr.Row():
        out_diabetes = gr.Text(label="Diabetes Risk")
        out_bmi      = gr.Text(label="BMI Info")

    out_explain = gr.Textbox(label="💬 AI Explanation", lines=6)

    btn.click(predict,
              inputs=[HighBP, HighChol, CholCheck, height_cm, weight_kg, Smoker, Stroke,
                      HeartDiseaseorAttack, PhysActivity, Fruits, Veggies,
                      HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth,
                      MentHlth, PhysHlth, DiffWalk, Sex, Age, Income, Education],
              outputs=[out_diabetes, out_bmi, out_explain])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b34f48c24604953f5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
